# **UNIVERSIDADE FEDERAL DO CEARA**
---
Disciplina: Introducao a analise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Julio Cesar Gama Feitosa Freitas - 583956
2.   Vitoria Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 10 — Spark SQL: queries interativas

## 🎯 Objetivo

Rodar as mesmas perguntas do Lab 8 (EDA), agora em Spark — sentindo a diferença de exploração interativa em memória.


In [43]:
# Instala o PySpark ANTES de montar a pipeline
!pip install pyspark --quiet

In [44]:
# Importa as bibliotecas e cria as pastas utilizadas pelo laboratorio
import os
import shutil
import duckdb

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)
os.makedirs("bigdata/silver", exist_ok=True)

# Abre uma conexao DuckDB (usada so para reconstruir a Silver)
con = duckdb.connect()

print("Ambiente preparado.")

Ambiente preparado.


## Passo 0 - Reconstruir a Silver

In [45]:
# Faz o upload dos CSVs brutos: customers_synthetic.csv e transactions_synthetic.csv
#from google.colab import files

uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [46]:
# Copia os CSVs enviados para a estrutura Raw do projeto
# Usa "in name" para tolerar sufixos que o Colab adiciona em uploads repetidos,
# como "customers_synthetic (1).csv"
for name in uploaded:
    if "customers_synthetic" in name:
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    elif "transactions_synthetic" in name:
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw preparados.")

Arquivos Raw preparados.


In [47]:
# Recria a Bronze de clientes e de transacoes com as mesmas regras do Lab 6
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

con.sql(f"""
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id, name, cpf, email, segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
""")

# O CASE converte explicitamente True/False (texto) para BOOLEAN.
con.sql(f"""
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id, customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type, status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE WHEN is_fraud = 'True' THEN true ELSE false END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
""")

con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()
con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [48]:
# Recria a Silver juntando transacoes com clientes e derivando as colunas de analise
con.sql("""
CREATE OR REPLACE TABLE silver_transactions AS
SELECT
  t.transaction_id, t.customer_id, t.amount, t.transaction_type,
  t.status, t.risk_score, t.is_fraud, t.ts,
  c.segment, c.credit_score,
  year(t.ts)  AS year,
  month(t.ts) AS month,
  day(t.ts)   AS day,
  dayofweek(t.ts) AS day_of_week,
  CASE
    WHEN t.amount < 100  THEN 'baixo'
    WHEN t.amount < 1000 THEN 'medio'
    ELSE 'alto'
  END AS amount_band
FROM bronze_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
""")

con.sql("SELECT COUNT(*) FROM silver_transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       100000 │
└──────────────┘



In [49]:
# Grava a Silver em Parquet apenas para o Spark ler a seguir nesta sessao
con.sql("""
COPY silver_transactions TO 'bigdata/silver/transactions_enriched.parquet' (FORMAT PARQUET)
""")

print("Parquet temporario pronto para o Spark ler.")

Parquet temporario pronto para o Spark ler.


## Setup do Spark

A partir daqui, os passos sao os mesmos das duas rotas do lab original - so muda de onde o Parquet veio.

In [50]:
# Cria a SparkSession local e le o Parquet da Silver
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("lab10").master("local[*]").getOrCreate()

df = spark.read.parquet("bigdata/silver/transactions_enriched.parquet")

## Passo 1 - Ver o schema e uma amostra

In [51]:
# Mostra a estrutura de colunas e tipos inferidos pelo Spark.
df.printSchema()

# Mostra as 5 primeiras linhas para uma conferencia visual rapida.
df.show(5)

root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: float (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- risk_score: float (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- ts: timestamp_ntz (nullable = true)
 |-- segment: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- day: long (nullable = true)
 |-- day_of_week: long (nullable = true)
 |-- amount_band: string (nullable = true)

+--------------+-----------+---------+----------------+--------+----------+--------+-------------------+--------+------------+----+-----+---+-----------+-----------+
|transaction_id|customer_id|   amount|transaction_type|  status|risk_score|is_fraud|                 ts| segment|credit_score|year|month|day|day_of_week|amount_band|
+--------------+-----------+---------+----------------+------

## Passo 2 - Criar a view temporaria (para usar SQL puro)

In [52]:
# Registra o DataFrame como uma view, permitindo consultas via spark.sql()
df.createOrReplaceTempView("silver_transactions")

## Passo 3 - Repetir a pergunta 1 do EDA (segmentacao) em SQL

In [53]:
# Score medio de credito por segmento, via SQL puro
spark.sql("""
SELECT segment, ROUND(AVG(credit_score), 1) AS score_medio, COUNT(*) AS total
FROM silver_transactions
GROUP BY segment
ORDER BY score_medio
""").show()

+---------+-----------+-----+
|  segment|score_medio|total|
+---------+-----------+-----+
| Standard|      643.9|29689|
|  Premium|      652.7|61156|
|High-Risk|      659.8| 9155|
+---------+-----------+-----+



## Passo 4 - A mesma pergunta, agora com a API de DataFrame

In [54]:
# Mesma agregacao do Passo 3, mas usando a API de DataFrame em vez de SQL.
from pyspark.sql import functions as F

(df.groupBy("segment")
   .agg(F.round(F.avg("credit_score"), 1).alias("score_medio"),
        F.count("*").alias("total"))
   .orderBy("score_medio")
   .show())

+---------+-----------+-----+
|  segment|score_medio|total|
+---------+-----------+-----+
| Standard|      643.9|29689|
|  Premium|      652.7|61156|
|High-Risk|      659.8| 9155|
+---------+-----------+-----+



**Compare:** os dois blocos (Passo 3 e 4) devem dar exatamente o mesmo resultado - sao dois jeitos de pedir a mesma coisa ao mesmo motor.

*(confira apos rodar as duas celulas acima)*

## Passo 5 - Medir o ganho do cache

In [55]:
# Mede o tempo de uma contagem filtrada antes de cachear o DataFrame
import time

t0 = time.time()
df.filter(df.is_fraud == True).count()
print("1a vez (sem cache):", time.time() - t0, "s")

# Marca o DataFrame para cache e forca a materializacao com uma acao (.count())
df.cache()
df.count()

# Mede a mesma contagem de novo - agora lendo do cache em memoria, nao do disco
t0 = time.time()
df.filter(df.is_fraud == True).count()
print("2a vez (com cache):", time.time() - t0, "s")

1a vez (sem cache): 0.1340632438659668 s
2a vez (com cache): 0.1073462963104248 s


**Esperado:** a segunda medicao tende a ser mais rapida - o DataFrame ja esta em memoria, sem reler do disco.

*(confira apos rodar a celula acima)*

## Passo 6 - Uma agregacao mais pesada: fraude por dia da semana

In [56]:
# Total de transacoes e de fraudes agrupados por dia da semana, usando a coluna ts
spark.sql("""
SELECT dayofweek(ts) AS dia_semana,
       COUNT(*) AS total,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes
FROM silver_transactions
GROUP BY dayofweek(ts)
ORDER BY dia_semana
""").show()

+----------+-----+-------+
|dia_semana|total|fraudes|
+----------+-----+-------+
|         1|14034|    257|
|         2|13940|    283|
|         3|14222|    262|
|         4|14338|    234|
|         5|14452|    263|
|         6|14484|    261|
|         7|14530|    273|
+----------+-----+-------+



## Checkpoint

- [ ] `df.show()` retornou dados reais da Silver
- [ ] SQL puro e API de DataFrame deram o mesmo resultado
- [ ] Voce mediu (mesmo que informalmente) o ganho de usar `.cache()`

---

**Proximo lab:** `DIA3_LAB11_DASHBOARD.md` - visualizar tudo isso.